# Agent-Based Muay Thai Strike Classifier

This notebook evaluates the LangGraph agent approach (from `basic_agent_explicit.py`) to serve as a comparison for the DSPy and baseline classifiers. It uses the same data splits and evaluation metrics.


## Section 1: Setup and Configuration


In [1]:
import json
from pathlib import Path
import os
from dotenv import load_dotenv
import random
import time
from typing import List, Dict, Tuple
import re
from collections import defaultdict

# Import agent from basic_agent_explicit
import sys
sys.path.append('agent')  # Add agent directory to path
from basic_agent_explicit import graph, llm
from langchain_core.messages import HumanMessage

# Load environment variables
load_dotenv()

# Data paths (same as other notebooks)
DATA_ROOT = Path("data/combos")
SHADOW_DIR = DATA_ROOT / "shadow"
HEAVY_BAG_DIR = DATA_ROOT / "heavy_bag"

# Validation split size (same as DSPy and baseline notebooks)
VAL_SPLIT_SIZE = 7

# Rate limiting for free tier
# Agent makes multiple API calls (analyze_video + potentially extract_pose_data)
# Use 60 seconds between videos to avoid rate limits
REQUEST_DELAY = 60

print("Setup complete!")
print(f"Data directories: {SHADOW_DIR}, {HEAVY_BAG_DIR}")
print(f"Rate limiting: {REQUEST_DELAY}s delay between videos")
print(f"Agent graph loaded successfully")


Setup complete!
Data directories: data\combos\shadow, data\combos\heavy_bag
Rate limiting: 60s delay between videos
Agent graph loaded successfully


## Section 2: Data Loading


In [2]:
def load_labels(json_path: Path) -> Dict[str, List[str]]:
    """Load true labels from JSON file."""
    with open(json_path, 'r') as f:
        return json.load(f)

def create_examples(labels: Dict[str, List[str]], video_dir: Path, dataset_name: str) -> List[Dict]:
    """Create example dictionaries from labels and video directory."""
    examples = []
    for video_id, strikes in labels.items():
        video_path = video_dir / f"{video_id}.mov"
        if video_path.exists():
            example = {
                'video_path': str(video_path),
                'strikes': strikes
            }
            examples.append(example)
        else:
            print(f"Warning: Video {video_path} not found")
    return examples

# Load labels from both datasets
shadow_labels = load_labels(SHADOW_DIR / "true_labels.json")
heavy_bag_labels = load_labels(HEAVY_BAG_DIR / "true_labels.json")

# Create examples
shadow_examples = create_examples(shadow_labels, SHADOW_DIR, "shadow")
heavy_bag_examples = create_examples(heavy_bag_labels, HEAVY_BAG_DIR, "heavy_bag")

print(f"Loaded {len(shadow_examples)} shadow examples")
print(f"Loaded {len(heavy_bag_examples)} heavy_bag examples")


Loaded 20 shadow examples
Loaded 19 heavy_bag examples


In [3]:
# Split data into train and validation
# Use the same random seed (42) as DSPy and baseline notebooks for reproducibility

random.seed(42)  # Same seed as other notebooks

# Shadow split
shadow_indices = list(range(len(shadow_examples)))
random.shuffle(shadow_indices)
shadow_val_indices = set(shadow_indices[:VAL_SPLIT_SIZE])
shadow_train = [shadow_examples[i] for i in range(len(shadow_examples)) if i not in shadow_val_indices]
shadow_val = [shadow_examples[i] for i in shadow_val_indices]

# Heavy bag split
heavy_bag_indices = list(range(len(heavy_bag_examples)))
random.shuffle(heavy_bag_indices)
heavy_bag_val_indices = set(heavy_bag_indices[:VAL_SPLIT_SIZE])
heavy_bag_train = [heavy_bag_examples[i] for i in range(len(heavy_bag_examples)) if i not in heavy_bag_val_indices]
heavy_bag_val = [heavy_bag_examples[i] for i in heavy_bag_val_indices]

# Combine train and validation sets
train_examples = shadow_train + heavy_bag_train
val_examples = shadow_val + heavy_bag_val

print(f"Training set: {len(train_examples)} examples ({len(shadow_train)} shadow + {len(heavy_bag_train)} heavy_bag)")
print(f"Validation set: {len(val_examples)} examples ({len(shadow_val)} shadow + {len(heavy_bag_val)} heavy_bag)")

# Display some statistics
all_strikes = set()
for ex in train_examples + val_examples:
    all_strikes.update(ex['strikes'])

print(f"\nUnique strike types: {sorted(all_strikes)}")
print(f"Total unique strike types: {len(all_strikes)}")


Training set: 25 examples (13 shadow + 12 heavy_bag)
Validation set: 14 examples (7 shadow + 7 heavy_bag)

Unique strike types: ['cross', 'jab', 'knee', 'lead_elbow', 'lead_hook', 'lead_uppercut', 'lowkick', 'rear_elbow', 'rear_hook', 'roundhouse', 'switch_kick', 'switch_knee', 'teep']
Total unique strike types: 13


## Section 3: Agent Classification Function


In [4]:
def classify_video_agent(video_path: str) -> List[str]:
    """
    Classify Muay Thai strikes in a video using the LangGraph agent.
    Returns a list of strike names in sequence.
    """
    try:
        # Invoke the agent
        result = graph.invoke(
            {"messages": [HumanMessage(content=f"Classify the strikes in this video: {video_path}")]}
        )
        
        # Extract response from the last message
        agent_response = result["messages"][-1].content
        
        # Handle different response formats
        # Response might be a string, list, or dict
        if isinstance(agent_response, str):
            response_text = agent_response
        elif isinstance(agent_response, list):
            # If response is a list of content blocks, extract text
            response_text = ""
            for item in agent_response:
                if isinstance(item, dict) and 'text' in item:
                    response_text += item['text']
                elif isinstance(item, str):
                    response_text += item
        elif isinstance(agent_response, dict):
            # If response is a dict, try to extract text
            response_text = agent_response.get('text', str(agent_response))
        else:
            response_text = str(agent_response)
        
        # Try to extract JSON array from response
        # First, try direct JSON parsing
        try:
            strikes = json.loads(response_text.strip())
            if isinstance(strikes, list):
                return strikes
        except (json.JSONDecodeError, ValueError):
            pass
        
        # If direct parsing fails, try to find JSON array in the text using regex
        json_match = re.search(r'\[.*?\]', response_text)
        if json_match:
            try:
                strikes = json.loads(json_match.group())
                if isinstance(strikes, list):
                    return strikes
            except json.JSONDecodeError:
                pass
        
        # If still no luck, return empty list
        print(f"Warning: Could not parse JSON from agent response: {response_text[:200]}")
        return []
        
    except Exception as e:
        print(f"Error classifying video {video_path}: {e}")
        return []

print("Agent classification function created!")


Agent classification function created!


## Section 4: Evaluation


In [5]:
# Evaluation metrics (same as DSPy and baseline notebooks)

def exact_match_metric(true_strikes: List[str], predicted_strikes: List[str]) -> bool:
    """Evaluate exact sequence match between predicted and true strikes."""
    return predicted_strikes == true_strikes

def sequence_accuracy_metric(true_strikes: List[str], predicted_strikes: List[str]) -> float:
    """Calculate sequence accuracy (partial credit for correct strikes in order)."""
    if not predicted_strikes or not true_strikes:
        return 0.0
    
    # Calculate positional accuracy
    min_len = min(len(predicted_strikes), len(true_strikes))
    if min_len == 0:
        return 0.0
    
    matches = sum(1 for i in range(min_len) if predicted_strikes[i] == true_strikes[i])
    return matches / len(true_strikes)

print("Evaluation metrics defined!")


Evaluation metrics defined!


In [6]:
# Evaluate on validation set
print(f"Evaluating on {len(val_examples)} validation examples...")
print(f"Using {REQUEST_DELAY}s delay between videos to avoid rate limits")
print(f"(Agent makes multiple API calls per video, so longer delays are needed)\n")

results = []
exact_matches = 0
total_accuracy = 0.0

for i, example in enumerate(val_examples):
    try:
        # Add rate limiting delay (except for first request)
        if i > 0:
            print(f"\nWaiting {REQUEST_DELAY}s to avoid rate limits...")
            time.sleep(REQUEST_DELAY)
        
        print(f"\n{'='*60}")
        print(f"Processing {i+1}/{len(val_examples)}: {Path(example['video_path']).name}")
        print(f"{'='*60}")
        
        # Classify video using agent
        predicted_strikes = classify_video_agent(example['video_path'])
        
        true_strikes = example['strikes']
        exact_match = exact_match_metric(true_strikes, predicted_strikes)
        accuracy = sequence_accuracy_metric(true_strikes, predicted_strikes)
        
        if exact_match:
            exact_matches += 1
        total_accuracy += accuracy
        
        results.append({
            'video_path': example['video_path'],
            'true_strikes': true_strikes,
            'predicted_strikes': predicted_strikes,
            'exact_match': exact_match,
            'sequence_accuracy': accuracy
        })
        
        print(f"True:  {true_strikes}")
        print(f"Pred:  {predicted_strikes}")
        print(f"Match: {'✓' if exact_match else '✗'} (Accuracy: {accuracy:.2%})")
            
    except Exception as e:
        print(f"Error processing {example['video_path']}: {e}\n")
        results.append({
            'video_path': example['video_path'],
            'true_strikes': example['strikes'],
            'predicted_strikes': [],
            'exact_match': False,
            'sequence_accuracy': 0.0,
            'error': str(e)
        })

print("\n" + "="*60)
print("Evaluation complete!")
print("="*60)


Evaluating on 14 validation examples...
Using 60s delay between videos to avoid rate limits
(Agent makes multiple API calls per video, so longer delays are needed)


Processing 1/14: 5.mov
[DEBUG] analyze_video returned: ["roundhouse"]...
True:  ['jab', 'cross', 'lead_hook', 'lead_hook']
Pred:  ['roundhouse']
Match: ✗ (Accuracy: 0.00%)

Waiting 60s to avoid rate limits...

Processing 2/14: 6.mov
[DEBUG] analyze_video returned: ["teep"]...
True:  ['jab', 'cross', 'switch_kick']
Pred:  ['teep']
Match: ✗ (Accuracy: 0.00%)

Waiting 60s to avoid rate limits...

Processing 3/14: 10.mov
[DEBUG] analyze_video returned: ["teep"]...
True:  ['jab', 'cross', 'switch_knee']
Pred:  []
Match: ✗ (Accuracy: 0.00%)

Waiting 60s to avoid rate limits...

Processing 4/14: 14.mov
[DEBUG] analyze_video returned: ["knee"]...
True:  ['lead_uppercut', 'rear_elbow', 'knee']
Pred:  ['knee']
Match: ✗ (Accuracy: 0.00%)

Waiting 60s to avoid rate limits...

Processing 5/14: 15.mov
[DEBUG] analyze_video returned: ["r

In [7]:
# Calculate and display metrics
exact_match_accuracy = exact_matches / len(val_examples) if val_examples else 0
average_sequence_accuracy = total_accuracy / len(val_examples) if val_examples else 0

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"Exact Match Accuracy: {exact_match_accuracy:.2%} ({exact_matches}/{len(val_examples)})")
print(f"Average Sequence Accuracy: {average_sequence_accuracy:.2%}")
print()

# Display some example predictions
print("Sample Predictions:")
print("-" * 60)
for i, result in enumerate(results[:5], 1):
    print(f"\nExample {i}:")
    print(f"  Video: {Path(result['video_path']).name}")
    print(f"  True:  {result['true_strikes']}")
    print(f"  Pred:  {result['predicted_strikes']}")
    print(f"  Match: {'✓' if result['exact_match'] else '✗'} (Accuracy: {result['sequence_accuracy']:.2%})")


EVALUATION RESULTS
Exact Match Accuracy: 0.00% (0/14)
Average Sequence Accuracy: 25.60%

Sample Predictions:
------------------------------------------------------------

Example 1:
  Video: 5.mov
  True:  ['jab', 'cross', 'lead_hook', 'lead_hook']
  Pred:  ['roundhouse']
  Match: ✗ (Accuracy: 0.00%)

Example 2:
  Video: 6.mov
  True:  ['jab', 'cross', 'switch_kick']
  Pred:  ['teep']
  Match: ✗ (Accuracy: 0.00%)

Example 3:
  Video: 10.mov
  True:  ['jab', 'cross', 'switch_knee']
  Pred:  []
  Match: ✗ (Accuracy: 0.00%)

Example 4:
  Video: 14.mov
  True:  ['lead_uppercut', 'rear_elbow', 'knee']
  Pred:  ['knee']
  Match: ✗ (Accuracy: 0.00%)

Example 5:
  Video: 15.mov
  True:  ['jab', 'lead_uppercut', 'lead_elbow', 'switch_knee']
  Pred:  ['rear_hook', 'roundhouse']
  Match: ✗ (Accuracy: 0.00%)


In [8]:
# Calculate per-strike accuracy
strike_stats = defaultdict(lambda: {'correct': 0, 'total': 0, 'predicted': 0})

for result in results:
    true_strikes = result['true_strikes']
    pred_strikes = result['predicted_strikes']
    
    # Count true strikes
    for strike in true_strikes:
        strike_stats[strike]['total'] += 1
    
    # Count predicted strikes
    for strike in pred_strikes:
        strike_stats[strike]['predicted'] += 1
    
    # Count correct predictions at each position
    min_len = min(len(true_strikes), len(pred_strikes))
    for i in range(min_len):
        if true_strikes[i] == pred_strikes[i]:
            strike_stats[true_strikes[i]]['correct'] += 1

print("\nPer-Strike Accuracy:")
print("-" * 60)
for strike in sorted(strike_stats.keys()):
    stats = strike_stats[strike]
    accuracy = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
    print(f"  {strike:20s}: {accuracy:6.2%} ({stats['correct']}/{stats['total']} correct, {stats['predicted']} predicted)")



Per-Strike Accuracy:
------------------------------------------------------------
  cross               : 33.33% (3/9 correct, 8 predicted)
  jab                 : 50.00% (5/10 correct, 9 predicted)
  knee                : 50.00% (1/2 correct, 6 predicted)
  lead_elbow          :  0.00% (0/3 correct, 2 predicted)
  lead_hook           : 20.00% (1/5 correct, 2 predicted)
  lead_uppercut       :  0.00% (0/3 correct, 2 predicted)
  rear_elbow          :  0.00% (0/3 correct, 1 predicted)
  rear_hook           :  0.00% (0/0 correct, 3 predicted)
  rear_uppercut       :  0.00% (0/0 correct, 2 predicted)
  roundhouse          :  0.00% (0/3 correct, 4 predicted)
  switch_kick         :  0.00% (0/4 correct, 2 predicted)
  switch_knee         :  0.00% (0/2 correct, 2 predicted)
  teep                : 50.00% (1/2 correct, 5 predicted)


## Section 5: Results and Comparison


In [9]:
# Save evaluation results (same format as DSPy and baseline notebooks)
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

# Save evaluation results
results_data = {
    'metrics': {
        'exact_match_accuracy': exact_match_accuracy,
        'average_sequence_accuracy': average_sequence_accuracy,
        'total_examples': len(val_examples),
        'exact_matches': exact_matches
    },
    'results': results,
    'strike_stats': {k: dict(v) for k, v in strike_stats.items()}
}

with open(output_dir / "agent_evaluation_results.json", 'w') as f:
    json.dump(results_data, f, indent=2)

print(f"Results saved to {output_dir / 'agent_evaluation_results.json'}")


Results saved to results\agent_evaluation_results.json


In [10]:
# Display summary
print("\n" + "=" * 60)
print("AGENT EVALUATION SUMMARY")
print("=" * 60)
print(f"Model: Gemini 2.5 Flash (LangGraph Agent)")
print(f"Approach: Agent-based with tools (analyze_video, extract_pose_data)")
print(f"Training examples: {len(train_examples)}")
print(f"Validation examples: {len(val_examples)}")
print(f"\nFinal Performance:")
print(f"  Exact Match Accuracy: {exact_match_accuracy:.2%}")
print(f"  Average Sequence Accuracy: {average_sequence_accuracy:.2%}")
print(f"\nResults saved to: {output_dir}/agent_evaluation_results.json")
print("\nNote: Compare these results with DSPy and baseline classifier results for evaluation.")
print("=" * 60)



AGENT EVALUATION SUMMARY
Model: Gemini 2.5 Flash (LangGraph Agent)
Approach: Agent-based with tools (analyze_video, extract_pose_data)
Training examples: 25
Validation examples: 14

Final Performance:
  Exact Match Accuracy: 0.00%
  Average Sequence Accuracy: 25.60%

Results saved to: results/agent_evaluation_results.json

Note: Compare these results with DSPy and baseline classifier results for evaluation.
